In [ ]:
from tqdm.notebook import trange
import pandas as pd



import utils
import visualization
import numpy as np
import os
from IPython.display import display, Video
import ctp_swap
from manavlib.common.params import (
    ExperimentParams,
    BaseDiscreteAgentParams,
    BaseAlgParams,
)
import manavlib.io.xml_io as new_io
from manavlib.gen.maps import reduce_cellsize

# Experiments

## Configuration

In [2]:
TASK_SUFFIX = "_task.xml"  # Set task file naming convention with a suffix (_task.xml).
MAP_FILE = "map.xml"  # Specify the name for the map file.
RESULT_FILE = "result.txt"  # Define a name for the result file.

TASK_DIR = (
    "../tasks/maze-32-32-4/"  # Define the directory path where task files are located.
)
CONFIG_FILE = "ctp_swap_config.xml"  # Select the algorithm (by choosing the config file) to be used (dec_tswap in this case).
RESULT_DIR = "./results/"  # Define the directory path for the result file.


AGENTS_FROM = 10  # Starting number of agents for the experiment.
AGENTS_TO = 60  # Maximum number of agents for the experiment.
AGENTS_STEP = 10  # Increment step for the number of agents in experiment run.
TASK_NUM = 5  # Number of tasks to run.

In [3]:
map_path = os.path.join(TASK_DIR, MAP_FILE)
config_path = os.path.join(TASK_DIR, CONFIG_FILE)

# Read experiment and algorithm parameters from the XML configuration file.
exp_params, alg_params = new_io.read_xml_config(config_path)


# Load map data from the XML map file.
h, w, cs, grid_map, obstacles = new_io.read_xml_map(map_path)

## Experiemnts Execution

In [4]:
result_path = os.path.join(RESULT_DIR, f"{RESULT_FILE}")
result_file = open(result_path, "w")
result_file.write(utils.Summary.header() + "\n")
result_file.close()
for task_id in (
    pbar_task := trange(
        TASK_NUM, bar_format="{desc:<20}{percentage:3.0f}%|{bar}{r_bar}"
    )
):
    pbar_task.set_description(f"{task_id+1} / {TASK_NUM} task")
    pbar_task.refresh()
    task_file = f"{task_id}{TASK_SUFFIX}"
    task_path = os.path.join(TASK_DIR, task_file)
    default_params, starts, goals, ag_params = new_io.read_xml_agents(task_path)
    

    # path_table = PathTable(grid_map, goals[:AGENTS_TO])
    ag_params_obj = ctp_swap.convert_agent_params(ag_params[0])

    (NavAlg, nav_params), (Planner, planner_params), (Follower, follower_params) = (
        utils.get_algorithms(alg_params)
    )

    planner_params_obj = ctp_swap.convert_alg_params(planner_params)
    # grid_map_obj = ctp_swap.GridMap(grid_map, cs)
    # planner = Planner(ag_params_obj, planner_params_obj, goals, grid_map_obj)


    


    for agents_num in (
        pbar_agent := trange(
            AGENTS_FROM,
            AGENTS_TO + 1,
            AGENTS_STEP,
            colour="#FFA500",
            bar_format="{desc:<20}{percentage:3.0f}%|{bar}{r_bar}",
        )
    ):
        
        vis_graph_obj = ctp_swap.VisibilityGraph(obstacles, goals[:agents_num, 0:2], 0.35)
        planner  = ctp_swap.VisibilityPlanner(ag_params_obj, planner_params_obj, goals[:agents_num, 0:2], vis_graph_obj)

        
        pbar_agent.set_description(
            f"Task {task_id}. {agents_num}/{AGENTS_TO} agents",
        )
        pbar_agent.refresh()
        simulation = utils.Simulation(
            starts[:agents_num, 0:2],
            goals[:agents_num, 0:2],
            grid_map,
            cs,
            obstacles,
            planner,
            agents_num,
            ag_params,
            alg_params,
            exp_params,
            False,
        )

        summary = simulation.run_experiment()
        # print(steps_log)
        
        
        # output_dir = "img"
        # output_filename = f"animated_trajectories_{agents_num}"
        # output_ext = "mp4"
        # output_path = os.path.join(output_dir, f"{output_filename}.{output_ext}")


        # grid_map_obj = ctp_swap.GridMap(grid_map, cs)
        # grid_map_obj.inflate(ag_params[0].size)
        # grid_map_inf = grid_map_obj.get_map()

        # visualization.draw(
        #     grid_map,
        #     grid_map_inf,
        #     cs,
        #     obstacles,
        #     goals,
        #     steps_log,
        #     ag_params,
        #     goal_log,
        #     neighbors_log,
        #     15,
        #     output_path,
        # )
        
        # display(Video(filename=output_path))
        
        result_file = open(result_path, "a")
        result_file.write(str(summary) + "\n")
        result_file.close()

                      0%|          | 0/5 [00:00<?, ?it/s]

                      0%|          | 0/6 [00:00<?, ?it/s]

                      0%|          | 0/6 [00:00<?, ?it/s]

                      0%|          | 0/6 [00:00<?, ?it/s]

                      0%|          | 0/6 [00:00<?, ?it/s]

                      0%|          | 0/6 [00:00<?, ?it/s]

## Preliminarily Results Processing

In [5]:
results = pd.read_table(result_path, skipinitialspace=True, sep=" ")
results.head()

,success,collision,collision_obst,makespan,flowtime,runtime,mean_groups,mean_groups_size,number
0,1,0,0,101.4,315.7,0.942,7.930,1.29,10
1,1,0,0,266.6,1141.4,4.665,17.888,1.13,20
2,1,0,0,122.2,999.9,3.755,22.417,1.35,30
3,1,0,0,194.2,1949.2,9.819,25.394,1.58,40
4,1,0,0,208.8,1836.3,15.691,29.283,1.72,50


In [6]:
gr_results = results.groupby("number")
sr = pd.DataFrame()
sr["success rate"] = gr_results["success"].mean()

success_results = results.drop(results[results.success < 1].index)

sr["flowtime"] = gr_results["flowtime"].mean()
sr["makespan"] = gr_results["makespan"].mean()
sr["runtime"] = gr_results["runtime"].mean()
sr["collisions"] = gr_results["collision"].sum() + gr_results["collision_obst"].sum()
sr_style = sr.style.format(
    {
        "success rate": "{:.0%}",
        "flowtime": "{:.1f}",
        "makespan": "{:.1f}",
        "runtime": "{:.3f}",
        "collisions": "{:}",
    }
)
sr_style

,success rate,flowtime,makespan,runtime,collisions
number,,,,,
10,100%,331.9,102.7,0.920,0
20,100%,705.1,149.2,2.770,0
30,100%,905.1,124.8,4.084,0
40,100%,2270.5,281.1,16.829,0
50,100%,1780.8,148.4,12.920,0
60,100%,1907.9,145.8,22.815,0
